In [25]:
import pandas as pd
import glob
import numpy as np
import matplotlib.pyplot as plt
from io import StringIO

In [26]:
FIGURES = '/lustre/ea-nrtmidas/users/3770/emulsion_stability/figures'

In [27]:
def filter_comments(file_path):
    """
    Generator to read a file and clubs lines that do not start 
    with '#' or '@'.
    """
    not_commented = []
    with open(file_path, 'r') as f:
        for line in f:
            if not line.strip().startswith('#') and not line.strip().startswith('@'):
                not_commented.append(line)
    return not_commented


In [28]:
def compute_mean_std(dfs, time_col, value_col):

    values_interp = []

    for df in dfs:
        t = df[time_col].values
        v = df[value_col].values

        values_interp.append(v)

    values_interp = np.array(values_interp)

    mean = values_interp.mean(axis=0)
    std = values_interp.std(axis=0)

    return t, mean, std

In [29]:
# plotting styles
config_marker = ['+', 'd', 's']
config_marker_dict = {'config-3':'+', 'config-2':'d', 'config-1':'s'}
line_styles = ['-','--', '-.', ':', (0, (5, 10)), (0, (3, 10, 1, 10)), (0, (3, 5, 1, 5, 1, 5)), (0, (5, 5)), (0, (5, 1))]

In [30]:
plt.style.use('seaborn-v0_8-whitegrid')  # clean base style

plt.rcParams.update({
    'font.size': 16,            # base font
    'axes.labelsize': 20,       # x/y labels
    'axes.titlesize': 18,       # subplot titles
    'xtick.labelsize': 16,      # x tick numbers
    'ytick.labelsize': 16,      # y tick numbers
    'legend.fontsize': 16,      # legend text
    'legend.title_fontsize': 15,
    'lines.markersize': 6,
    'figure.titlesize': 20
})

# Radial Distribution Function Analysis

In [31]:
'''
get labels from the rdf file, which are used as column names in the dataframe
'''
def get_rdf_labels(filename):
    labels = ["r (nm)"] # first column is always the bin radius

    with open(filename, 'r') as f:
        for line in f:
            line = line.strip()

            if line.startswith('@ s') and 'legend' in line:
                parts = line.split()
                labels.append(parts[-1][1:-1]) # removing " " from the label

    return labels

## Surfactant 120

In [32]:
dir = '/lustre/ea-nrtmidas/users/3770/emulsion_stability/data/raw/self_assembly_polymer_surfactant120_toluene_water_20260402-185803/production_NVT'
ENSEMBLE = dir.split('/')[-1]
SUB_DIR = dir.split('/')[-2] 

In [33]:
file_rdf = 'rdf_analysis/*_MB.xvg'
ref_chem = file_rdf.split('/')[-1].split('_')[-1][:-4]
all_rdf_files = glob.glob(f'{dir}/*/{file_rdf}')
dfs_rdf_list = []
config_list = []
for f in all_rdf_files:
    config = f.split('/')[-3]
    config_list.append(config)
    # Use StringIO to treat the filtered lines as a file for read_csv
    removed_comments = filter_comments(f'{f}')
    file_removed_comments = StringIO('\n'.join(removed_comments))
    names_label = get_rdf_labels(f)
    df = pd.read_csv(file_removed_comments, header=None, sep='\s+', names=names_label)
    dfs_rdf_list.append(df)

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(7,6), sharex=True)
interval_thermo = 1

# # RDF - E2
# t, mean, std = compute_mean_std(
#     dfs_rdf_list,
#     'r (nm)',
#     'Poly_E2'
# )
# axes.plot(t[interval_thermo::interval_thermo], mean[interval_thermo::interval_thermo], color='tab:blue', label='Poly_MB - Poly_E2')
# axes.fill_between(
#     t[interval_thermo::interval_thermo],
#     mean[interval_thermo::interval_thermo] - std[interval_thermo::interval_thermo],
#     mean[interval_thermo::interval_thermo] + std[interval_thermo::interval_thermo],
#     color='tab:blue',
#     alpha=0.3
# )

# RDF - Toluene
t, mean, std = compute_mean_std(
    dfs_rdf_list,
    'r (nm)',
    'Tolue'
)
axes.plot(t[interval_thermo::interval_thermo], mean[interval_thermo::interval_thermo], color='tab:orange', label='Poly_MB - Toluene')
axes.fill_between(
    t[interval_thermo::interval_thermo],
    mean[interval_thermo::interval_thermo] - std[interval_thermo::interval_thermo],
    mean[interval_thermo::interval_thermo] + std[interval_thermo::interval_thermo],
    color='tab:orange',
    alpha=0.3
)

# RDF - Surfactant Head
t, mean, std = compute_mean_std(
    dfs_rdf_list,
    'r (nm)',
    'Head_1_2'
    
)
axes.plot(t[interval_thermo::interval_thermo], mean[interval_thermo::interval_thermo], color='tab:blue', label='Poly_MB - Head12')
axes.fill_between(
    t[interval_thermo::interval_thermo],
    mean[interval_thermo::interval_thermo] - std[interval_thermo::interval_thermo],
    mean[interval_thermo::interval_thermo] + std[interval_thermo::interval_thermo],
    color='tab:blue',
    alpha=0.3
)

axes.set(xlabel='r (nm)', ylabel='g(r)')
axes.grid(alpha=0.2)
axes.axhline(1, ls='--', color='black')
fig.legend(bbox_to_anchor=(1.0, 1.0))
plt.tight_layout()
fig.savefig(f'{FIGURES}/{SUB_DIR}/{ENSEMBLE}/RDF_Toluene_HEAD12.png', bbox_inches="tight")

## Surfactant 180

In [35]:
dir = '/lustre/ea-nrtmidas/users/3770/emulsion_stability/data/raw/self_assembly_polymer_surfactant180_toluene_water_20260401-163540/production_NVT'
ENSEMBLE = dir.split('/')[-1]
SUB_DIR = dir.split('/')[-2] 

In [36]:
file_rdf = 'rdf_analysis/*_MB.xvg'
ref_chem = file_rdf.split('/')[-1].split('_')[-1][:-4]
all_rdf_files = glob.glob(f'{dir}/*/{file_rdf}')
dfs_rdf_list = []
config_list = []
for f in all_rdf_files:
    config = f.split('/')[-3]
    config_list.append(config)
    # Use StringIO to treat the filtered lines as a file for read_csv
    removed_comments = filter_comments(f'{f}')
    file_removed_comments = StringIO('\n'.join(removed_comments))
    names_label = get_rdf_labels(f)
    df = pd.read_csv(file_removed_comments, header=None, sep='\s+', names=names_label)
    dfs_rdf_list.append(df)

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(7,6), sharex=True)
interval_thermo = 1

# # RDF - E2
# t, mean, std = compute_mean_std(
#     dfs_rdf_list,
#     'r (nm)',
#     'Poly_E2'
# )
# axes.plot(t[interval_thermo::interval_thermo], mean[interval_thermo::interval_thermo], color='tab:blue', label='Poly_MB - Poly_E2')
# axes.fill_between(
#     t[interval_thermo::interval_thermo],
#     mean[interval_thermo::interval_thermo] - std[interval_thermo::interval_thermo],
#     mean[interval_thermo::interval_thermo] + std[interval_thermo::interval_thermo],
#     color='tab:blue',
#     alpha=0.3
# )

# RDF - Toluene
t, mean, std = compute_mean_std(
    dfs_rdf_list,
    'r (nm)',
    'Tolue'
)
axes.plot(t[interval_thermo::interval_thermo], mean[interval_thermo::interval_thermo], color='tab:orange', label='Poly_MB - Toluene')
axes.fill_between(
    t[interval_thermo::interval_thermo],
    mean[interval_thermo::interval_thermo] - std[interval_thermo::interval_thermo],
    mean[interval_thermo::interval_thermo] + std[interval_thermo::interval_thermo],
    color='tab:orange',
    alpha=0.3
)

# RDF - Surfactant Head
t, mean, std = compute_mean_std(
    dfs_rdf_list,
    'r (nm)',
    'Head_1_2'
    
)
axes.plot(t[interval_thermo::interval_thermo], mean[interval_thermo::interval_thermo], color='tab:blue', label='Poly_MB - Head12')
axes.fill_between(
    t[interval_thermo::interval_thermo],
    mean[interval_thermo::interval_thermo] - std[interval_thermo::interval_thermo],
    mean[interval_thermo::interval_thermo] + std[interval_thermo::interval_thermo],
    color='tab:blue',
    alpha=0.3
)

axes.set(xlabel='r (nm)', ylabel='g(r)')
axes.grid(alpha=0.2)
axes.axhline(1, ls='--', color='black')
fig.legend(bbox_to_anchor=(1.0, 1.0))
plt.tight_layout()
fig.savefig(f'{FIGURES}/{SUB_DIR}/{ENSEMBLE}/RDF_Toluene_HEAD12.png', bbox_inches="tight")

## Comparison between different Surfactant count using RDF

In [38]:
dir0 = '/lustre/ea-nrtmidas/users/3770/emulsion_stability/data/raw/self_assembly_polymer_surfactant60_toluene_water_20260401-143905/production_NVT'
dir1 = '/lustre/ea-nrtmidas/users/3770/emulsion_stability/data/raw/self_assembly_polymer_surfactant120_toluene_water_20260402-185803/production_NVT'
dir2 = '/lustre/ea-nrtmidas/users/3770/emulsion_stability/data/raw/self_assembly_polymer_surfactant180_toluene_water_20260401-163540/production_NVT'

In [ ]:
file_rdf = 'rdf_analysis/*_MB.xvg'
ref_chem = file_rdf.split('/')[-1].split('_')[-1][:-4]
all_rdf_files_0 = glob.glob(f'{dir0}/*/{file_rdf}')
all_rdf_files_1 = glob.glob(f'{dir1}/*/{file_rdf}')
all_rdf_files_2 = glob.glob(f'{dir2}/*/{file_rdf}')
all_rdf_files = all_rdf_files_0+all_rdf_files_1+all_rdf_files_2
dfs_rdf_list_60 = []
dfs_rdf_list_120 = []
dfs_rdf_list_180 = []
config_list = []
for f in all_rdf_files:
    print(f)
    config = f.split('/')[-3]
    config_list.append(config)
    # Use StringIO to treat the filtered lines as a file for read_csv
    removed_comments = filter_comments(f'{f}')
    file_removed_comments = StringIO('\n'.join(removed_comments))
    names_label = get_rdf_labels(f)
    df = pd.read_csv(file_removed_comments, header=None, sep='\s+', names=names_label)
    if('surfactant60' in f):
         dfs_rdf_list_60.append(df)
    elif('surfactant120' in f):
        dfs_rdf_list_120.append(df)
    elif('surfactant180' in f):
        dfs_rdf_list_180.append(df)
    else:
        continue

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(7,6), sharex=True)
interval_thermo = 1

# # Surfactant 60
# t, mean, std = compute_mean_std(dfs_rdf_list_60, 'r (nm)','Head_1_2')
# axes.plot(t[interval_thermo::interval_thermo], mean[interval_thermo::interval_thermo], color='tab:red', label='Surfactant 60')
# axes.fill_between(
#     t[interval_thermo::interval_thermo],
#     mean[interval_thermo::interval_thermo] - std[interval_thermo::interval_thermo],
#     mean[interval_thermo::interval_thermo] + std[interval_thermo::interval_thermo],
#     color='tab:red',
#     alpha=0.3
# )

# Surfactant 120
t, mean, std = compute_mean_std(dfs_rdf_list_120, 'r (nm)','Head_1_2')
axes.plot(t[interval_thermo::interval_thermo], mean[interval_thermo::interval_thermo], color='tab:orange', label='Surfactant 120')
axes.fill_between(
    t[interval_thermo::interval_thermo],
    mean[interval_thermo::interval_thermo] - std[interval_thermo::interval_thermo],
    mean[interval_thermo::interval_thermo] + std[interval_thermo::interval_thermo],
    color='tab:orange',
    alpha=0.3
)

# Surfactant 180
t, mean, std = compute_mean_std(dfs_rdf_list_180, 'r (nm)','Head_1_2')
axes.plot(t[interval_thermo::interval_thermo], mean[interval_thermo::interval_thermo], color='tab:blue', label='Surfactant 180')
axes.fill_between(
    t[interval_thermo::interval_thermo],
    mean[interval_thermo::interval_thermo] - std[interval_thermo::interval_thermo],
    mean[interval_thermo::interval_thermo] + std[interval_thermo::interval_thermo],
    color='tab:blue',
    alpha=0.3
)

axes.set(xlabel='r (nm)', ylabel='g(r)')
axes.grid(alpha=0.2)
axes.axhline(1, ls='--', color='black')
fig.legend(bbox_to_anchor=(1.0, 1.0))
plt.tight_layout()
fig.savefig(f'{FIGURES}/RDF_vary_surfactant_MB-Head12.png', bbox_inches="tight")

# Comparison between sufactant systems using final number of clusters

## System with size of 23^3 nm^3

In [19]:
dir = '/lustre/ea-nrtmidas/users/3770/emulsion_stability/data/raw/self_assembly_*/production_NVT'

In [ ]:
file_pattern = 'cluster/my_num_clusters.xvg'
all_files = glob.glob(f'{dir}/*/{file_pattern}')
configs_data_list = []
config_list = []
for f in all_files:
    config = f.split('/')[-3]
    config_list.append(config)
    # Use StringIO to treat the filtered lines as a file for read_csv
    removed_comments = filter_comments(f'{f}')
    file_removed_comments = StringIO('\n'.join(removed_comments))
    df = pd.read_csv(file_removed_comments, header=None, sep='\s+', names=['Time (ps)', 'Number of Clusters'])
    configs_data_list.append(df)

In [ ]:
cluster_dist_surfactant = {}
for i,df in enumerate(configs_data_list[:]):
    file = all_files[i]
    surf = int(file.split('/')[-5].split('_')[3][10:])
    if(len(df) == 0):
        print(file)
        continue
    else:
        if(surf not in cluster_dist_surfactant.keys()):
            cluster_dist_surfactant[surf] = [np.array(df['Number of Clusters'])[-1]]
        else:
            cluster_dist_surfactant[surf].append(df['Number of Clusters'].iloc[-1])

In [ ]:
cluster_dist_surfactant

In [ ]:
fig, ax = plt.subplots()

labels = np.array(list(cluster_dist_surfactant.keys()))
means = np.array([np.mean(v) for v in cluster_dist_surfactant.values()])
stds = np.array([np.std(v) for v in cluster_dist_surfactant.values()])

# sort labels
idx = np.argsort(labels)
labels_sorted = labels[idx]
means_sorted = means[idx]
stds_sorted = stds[idx]

x = np.arange(len(labels_sorted))

# bar plot (averages)
ax.bar(x, means_sorted, capsize=5, alpha=0.6)

# markers for configs
markers = ['+', 'd', 's']  # config-1, 2, 3

for i, key in enumerate(labels_sorted):
    values = cluster_dist_surfactant[key]
    
    for j, val in enumerate(values):
        x_jitter = (j - 1) * 0.2  # spreads points slightly
        ax.scatter(x[i] + x_jitter, val,
                   marker=markers[j],
                   color='black',
                   s=70)

# legend for configs
for j, m in enumerate(markers):
    ax.scatter([], [], marker=m, color='black', label=f'Config-{j+1}')

ax.legend()

# labels
ax.set_xticks(x)
ax.set_xticklabels(labels_sorted)
ax.set_ylabel('Average number of clusters')
ax.set_xlabel('Surfactant Count')

plt.show()
fig.savefig(f'{FIGURES}/clusters_vary_surfactant.png', bbox_inches="tight")

## System with size of 34.5^3 nm^3

In [10]:
dir = '/lustre/ea-nrtmidas/users/3770/emulsion_stability/data/raw/box_var_self_assembly_polymer_surfactant_toluene_water_20260426-192641/production_NVT'

In [21]:
file_pattern = 'cluster/my_num_clusters.xvg'
all_files = glob.glob(f'{dir}/*/{file_pattern}')
configs_data_list = []
config_list = []
for f in all_files:
    config = f.split('/')[-3]
    config_list.append(config)
    # Use StringIO to treat the filtered lines as a file for read_csv
    removed_comments = filter_comments(f'{f}')
    file_removed_comments = StringIO('\n'.join(removed_comments))
    df = pd.read_csv(file_removed_comments, header=None, sep='\s+', names=['Time (ps)', 'Number of Clusters'])
    configs_data_list.append(df)

In [22]:
cluster_dist_surfactant_scaled = {}
for i,df in enumerate(configs_data_list[:]):
    file = all_files[i]
    surf = int(file.split('/')[-3].split('_.')[3].split('-')[-1])
    if(len(df) == 0):
        print(file)
        continue
    else:
        if(surf not in cluster_dist_surfactant_scaled .keys()):
            cluster_dist_surfactant_scaled [surf] = [np.array(df['Number of Clusters'])[-1]]
        else:
            cluster_dist_surfactant_scaled [surf].append(df['Number of Clusters'].iloc[-1])

In [ ]:
cluster_dist_surfactant_scaled 

In [ ]:
fig, ax = plt.subplots()

labels = np.array(list(cluster_dist_surfactant_scaled .keys()))
means = np.array([np.mean(v) for v in cluster_dist_surfactant_scaled .values()])
stds = np.array([np.std(v) for v in cluster_dist_surfactant_scaled .values()])

# sort labels
idx = np.argsort(labels)
labels_sorted = labels[idx]
means_sorted = means[idx]
stds_sorted = stds[idx]

x = np.arange(len(labels_sorted))

# bar plot (averages)
ax.bar(x, means_sorted, capsize=5, alpha=0.6)

# labels
ax.set_xticks(x)
ax.set_xticklabels(labels_sorted)
ax.set_ylabel('Average number of clusters')
ax.set_xlabel('Surfactant Count')

plt.show()
fig.savefig(f'{FIGURES}/clusters_vary_surfactant_scaled_box.png', bbox_inches="tight")